# 5 - Hiérarchies de colonnes

Une hiérarchie de menu (group-options, arbre de sélection de profondeur arbitraire)
est **une chaîne de colonnes** de la `fact_table`, déclarée dans `metadata.parent_name`
(cf. `specification-bdd.md`, §2.5). Il n'existe **aucune table auxiliaire** : la
parente de `commune` est `departement`, la parente de `departement` est `region`, et
la hiérarchie des *valeurs* est déjà dans la fact table
(`SELECT DISTINCT region, departement, commune`).

Invariants validés à l'écriture :

- la colonne parente existe (dans le DataFrame à la construction, dans `metadata` lors
  d'une correction) ;
- le graphe des `parent_name` est une **forêt** (pas de cycle, une seule parente par
  colonne), détecté par un parcours de proche en proche ;
- une colonne d'une hiérarchie est **catégorielle** ; si elle ne l'est pas par le
  seuil, elle est **forcée** à `True` avec un avertissement, comme le ferait
  `categorical_overrides`.

**Convention pour un arbre irrégulier** (feuilles à des profondeurs différentes) : les
niveaux absents sont `NULL`. Le constructeur d'arbre s'arrête au premier niveau `NULL`
et ne répète jamais la valeur du niveau supérieur (cela ferait apparaître un faux
nœud).

### Table des matières

0. [Importation des modules](#s0)
1. [Données synthétiques : une hiérarchie géographique irrégulière](#s1)
2. [Déclarer la hiérarchie via `hierarchies` à la construction](#s2)
   - [Forçage catégoriel d'une colonne de hiérarchie](#s2_1)
3. [Alternative : `parent_name` via `column_metadata`](#s3)
4. [Reconstituer les chaînes avec `get_column_hierarchies`](#s4)
5. [Construire un arbre de menu par `SELECT DISTINCT`](#s5)
6. [Corriger la hiérarchie sur une base existante](#s6)
7. [Supprimer une colonne parente : refus, puis `cascade=True`](#s7)
8. [Cas d'erreur validés à l'écriture](#s8)

## 0. Importation des modules <a id="s0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import sys
import warnings

import narwhals as nw
import polars as pl

# Ajout du chemin vers le package
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.operations import DatabaseDeleter, DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder, SchemaBuilder
from dt_ducklake_manager.utils import get_column_hierarchies

## 1. Données synthétiques : une hiérarchie géographique irrégulière <a id="s1"></a>

Trois niveaux de colonnes : `region` -> `departement` -> `commune`. La dernière ligne
illustre un arbre **irrégulier** : ce département n'a pas de commune renseignée
(`commune = None`), comme le prévoit la convention `NULL`.

In [ ]:
df = pl.DataFrame(
    {
        "id": [1, 2, 3, 4, 5],
        "date": ["2026-01-01"] * 5,
        "region": [
            "Bretagne",
            "Bretagne",
            "Bretagne",
            "Ile-de-France",
            "Ile-de-France",
        ],
        "departement": [
            "Finistere",
            "Finistere",
            "Morbihan",
            "Paris",
            "Essonne",
        ],
        "commune": ["Brest", "Quimper", "Vannes", "Paris", None],
        "value": [12.5, 8.1, 6.3, 40.2, 5.0],
    }
)
df

## 2. Déclarer la hiérarchie via `hierarchies` à la construction <a id="s2"></a>

`SchemaBuilder` / `DuckLakeTablesBuilder` acceptent un paramètre dédié
`hierarchies: dict[str, str]` (colonne enfant -> colonne parente). Il est validé et
écrit dans `metadata.parent_name` en une seule fois, sans table auxiliaire.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    builder = DuckLakeTablesBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "departement", "departement": "region"},
        dataset_label="Consommation régionale",
    )

builder.build_schema()

builder.conn.execute(
    "SELECT name, parent_name, is_categorical FROM metadata ORDER BY name"
).pl()

`region`, `departement` et `commune` portent chacune le nom de leur colonne parente
(`NULL` pour `region`, la racine). Les trois colonnes sont déjà catégorielles sous ce
seuil.

### Forçage catégoriel d'une colonne de hiérarchie <a id="s2_1"></a>

Avec un seuil plus bas, `commune` (3 modalités) resterait catégorielle mais
`departement` franchirait le seuil s'il avait plus de deux modalités. Voici le cas où
une colonne de la hiérarchie n'est **pas** catégorielle par le seuil : elle est forcée
à `True`, exactement comme `categorical_overrides` le ferait.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    forced_builder = SchemaBuilder(
        df,
        categorical_threshold=1,  # aucune colonne texte n'est catégorielle par seuil
        primary_keys=["id"],
        hierarchies={"commune": "departement", "departement": "region"},
    )
    forced_metadata = forced_builder.create_metadata_table()

print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
forced_metadata.filter(nw.col("name").is_in(["region", "departement", "commune"]))[
    ["name", "is_categorical"]
]

## 3. Alternative : `parent_name` via `column_metadata` <a id="s3"></a>

`parent_name` peut aussi être renseigné colonne par colonne dans `column_metadata`
(prompt 3), au même titre que `unit` ou `default_aggregation`. Les deux sources
doivent être **cohérentes** : une contradiction lève une `ValueError` explicite.

In [ ]:
# Équivalent au paramètre hierarchies, exprimé via column_metadata
alt_builder = SchemaBuilder(
    df,
    categorical_threshold=10,
    primary_keys=["id"],
)
alt_metadata = alt_builder.create_metadata_table(
    column_metadata={"commune": {"parent_name": "departement"}}
)
alt_metadata.filter(nw.col("name") == "commune")[["name", "parent_name"]]

In [ ]:
# hierarchies et column_metadata en désaccord sur la parente de 'commune'
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "region"},
    ).create_metadata_table(column_metadata={"commune": {"parent_name": "departement"}})
except ValueError as exc:
    print("hiérarchies contradictoires ->", exc)

## 4. Reconstituer les chaînes avec `get_column_hierarchies` <a id="s4"></a>

Fonction de référence pour les couches API : elle relit `metadata.parent_name` et
retourne une liste de chaînes racine -> feuille, une par colonne feuille.

In [ ]:
get_column_hierarchies(builder.conn, schema=builder.schema)

## 5. Construire un arbre de menu par `SELECT DISTINCT` <a id="s5"></a>

La hiérarchie des *valeurs* est déjà dans la fact table : un `SELECT DISTINCT` sur la
chaîne de colonnes suffit à obtenir l'arbre complet, sans aucune jointure. Convention
pour l'arbre irrégulier : on s'arrête au premier niveau `NULL` et on ne répète jamais
la valeur du niveau supérieur.

In [ ]:
chain = get_column_hierarchies(builder.conn, schema=builder.schema)[0]
rows = builder.conn.execute(
    f"SELECT DISTINCT {', '.join(chain)} FROM {builder._qualified('fact_table')}"
    f" ORDER BY {', '.join(chain)}"
).fetchall()


def build_menu_tree(chain: list[str], rows: list[tuple]) -> dict:
    """Build a nested {label: subtree} menu from root-to-leaf DISTINCT rows.

    Stops at the first NULL level in a row and never repeats the parent's own
    label as a fake child node.
    """
    tree: dict = {}
    for row in rows:
        node = tree
        for value in row:
            if value is None:
                break
            node = node.setdefault(value, {})
    return tree


build_menu_tree(chain, rows)

La branche `Ile-de-France -> Essonne` s'arrête bien à `departement` : aucun faux nœud
`None` ni répétition de `"Essonne"` au niveau `commune`.

## 6. Corriger la hiérarchie sur une base existante <a id="s6"></a>

`update_column_metadata(column, parent_name=...)` déclare ou corrige un lien de
hiérarchie sans reconstruire la base. Il valide l'existence de la colonne parente dans
`metadata` et l'absence de cycle sur le graphe courant, et force catégorielles les deux
extrémités du lien si besoin.

In [ ]:
updater = DatabaseUpdater(connection=builder.conn, categorical_threshold=10)

# 'value' n'est pas catégorielle : la déclarer enfant de 'commune' force son statut
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    updater.update_column_metadata("value", parent_name="commune")

print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
builder.conn.execute(
    "SELECT name, parent_name, is_categorical FROM metadata"
    " WHERE name IN ('value', 'commune')"
).pl()

In [ ]:
# Une modification créant un cycle est refusée (commune -> value -> commune)
try:
    updater.update_column_metadata("commune", parent_name="value")
except ValueError as exc:
    print("cycle refusé ->", exc)

## 7. Supprimer une colonne parente : refus, puis `cascade=True` <a id="s7"></a>

`DatabaseDeleter.delete_columns` refuse par défaut de supprimer une colonne qui est la
parente d'une autre colonne : cela orphelinerait la hiérarchie. `cascade=True` autorise
la suppression et détache les enfants (`parent_name` mis à `NULL`, avec un
avertissement).

In [ ]:
deleter = DatabaseDeleter(connection=builder.conn)

# Refus par défaut : 'departement' est la parente de 'commune'
result = deleter.delete_columns(["departement"], use_transaction=False)
print("sans cascade ->", result)

In [ ]:
# cascade=True : suppression autorisée, 'commune' est détachée
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = deleter.delete_columns(
        ["departement"], use_transaction=False, cascade=True
    )

print("avec cascade ->", result)
print([str(w.message) for w in caught if issubclass(w.category, UserWarning)])
builder.conn.execute(
    "SELECT name, parent_name FROM metadata WHERE name = 'commune'"
).pl()

## 8. Cas d'erreur validés à l'écriture <a id="s8"></a>

In [ ]:
# 8.1 Auto-référence (une colonne ne peut pas être sa propre parente)
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"region": "region"},
    ).create_metadata_table()
except ValueError as exc:
    print("auto-référence         ->", exc)

# 8.2 Cycle à deux colonnes (A -> B -> A)
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"region": "departement", "departement": "region"},
    ).create_metadata_table()
except ValueError as exc:
    print("cycle à deux colonnes   ->", exc)

# 8.3 Colonne parente inexistante dans le DataFrame
try:
    SchemaBuilder(
        df,
        categorical_threshold=10,
        primary_keys=["id"],
        hierarchies={"commune": "not_a_column"},
    )
except ValueError as exc:
    print("parente inexistante     ->", exc)

# 8.4 update_column_metadata : parente absente de la table metadata
try:
    updater.update_column_metadata("value", parent_name="not_a_column")
except ValueError as exc:
    print("parente absente (update) ->", exc)